# Q-TRACE: Colab CPU/GPU benchmark (optional remote compute)

This notebook measures whether Colab speeds up Q-TRACE's simulation hot path. That hot path is 4-qubit Aer density-matrix probabilities, folded at λ = 1, 3, 5, under registered Phase 5C noise. See `docs/REMOTE_COMPUTE.md` for the full workflow.

**Safety rules (enforced by the code):**
- **Code:** a *pinned, pushed* commit is cloned read-only. Nothing is ever pushed from Colab.
- **Data:** none is needed. The benchmark uses deterministic synthetic bindings, so no transaction data, controller partition or sealed lockbox material is uploaded.
- **Results:** one hashed zip comes back and is imported locally into `results/remote/<job_id>/`. The importer verifies every hash and the commit, and never overwrites anything.

**Runtime choice:** use a **T4** (≈ 1.2 CU/h; natively supported) or a CPU runtime. Avoid H100/G4 (not covered by the GPU wheel).

**Cost:** about 10–25 min on a GPU runtime and about 15–40 min on a CPU runtime. Multiply the elapsed time printed at the end by the compute-unit rate shown under *Runtime → View resources*.

**Setup (once):**
1. Create a GitHub fine-grained token with **read-only Contents** access to this repository only.
2. Add it to Colab **Secrets** as `GITHUB_TOKEN`, with notebook access enabled.

In [ ]:
# ---- parameters -------------------------------------------------------------------------
import re

PINNED_COMMIT = "REPLACE_WITH_40_CHAR_COMMIT"  # a pushed commit containing scripts/remote/
REPO = "MayankKumarPokhriyal/q-trace"
ROWS = "2048,8192,32768"  # bindings per call (production uses 4,096-row chunks)
assert re.fullmatch(r"[0-9a-f]{40}", PINNED_COMMIT), "set PINNED_COMMIT to a full commit"

In [ ]:
# ---- runtime facts (recorded again inside the benchmark JSON) ----------------------------
import shutil
import subprocess
import time

T0 = time.time()
smi = shutil.which("nvidia-smi")
HAS_GPU = smi is not None and subprocess.run([smi], capture_output=True).returncode == 0
print("GPU runtime:", HAS_GPU)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || true
!lscpu | grep -E "Model name|^CPU\(s\)" ; free -g | head -2 ; python3 --version

In [ ]:
# ---- clone the pinned commit with a read-only token from Colab Secrets (never printed) ----
import os
import pathlib

from google.colab import userdata

askpass = pathlib.Path("/tmp/qtrace_askpass.sh")
askpass.write_text('#!/bin/sh\nprintf "%s" "$QTRACE_GIT_TOKEN"\n')
askpass.chmod(0o700)
env = dict(
    os.environ,
    GIT_ASKPASS=str(askpass),
    GIT_TERMINAL_PROMPT="0",
    QTRACE_GIT_TOKEN=userdata.get("GITHUB_TOKEN"),
)
dest = "/content/q-trace"
url = f"https://x-access-token@github.com/{REPO}.git"
subprocess.run(["rm", "-rf", dest], check=True)
subprocess.run(["git", "clone", "--quiet", "--no-checkout", url, dest], env=env, check=True)
subprocess.run(["git", "-C", dest, "checkout", "--quiet", "--detach", PINNED_COMMIT], check=True)
del env
askpass.unlink()


def git(*args: str) -> str:
    return subprocess.run(["git", "-C", dest, *args], capture_output=True, text=True).stdout


head, dirty = git("rev-parse", "HEAD").strip(), git("status", "--porcelain")
assert head == PINNED_COMMIT and not dirty, (head, dirty)
subprocess.run(["git", "-C", dest, "remote", "remove", "origin"], check=True)  # cannot push
print("checked out", head)

In [ ]:
# ---- environment from the lock file: uv-managed CPython 3.12 (Colab default is 3.13) -----
# Colab sets UV_SYSTEM_PYTHON / UV_CONSTRAINT / ... which break `uv sync`; clear them here.
import os

for k in [k for k in os.environ if k.startswith("UV_")]:
    os.environ.pop(k)
os.environ["UV_PYTHON_PREFERENCE"] = "only-managed"
%cd /content/q-trace
!pip -q install uv
!uv python install 3.12
!uv sync --frozen --python 3.12 --quiet
!.venv/bin/python -c "import sys, qiskit_aer as a; print(sys.version.split()[0], a.__version__)"
# If this cell fails on the current image: Runtime -> Change runtime type -> version 2026.07
# (Ubuntu 22.04, Python 3.12, CUDA 12.8), then rerun from the top.

In [ ]:
# ---- GPU only: replace qiskit-aer by the CUDA-11 GPU build of the SAME version ------------
# Aer 0.17.2 has no CUDA-12 GPU wheel; qiskit-aer-gpu-cu11 (same qiskit_aer import) ships its
# CUDA 11 user-space libraries via pip. Native on T4 (sm_75) and A100 (sm_80); L4 (sm_89)
# should load sm_86 code; H100 / G4 are NOT covered by the wheel.
# Never call `uv run` or `uv sync` after this cell: they would restore the CPU-only lock.
PY = ".venv/bin/python"
if HAS_GPU:
    ver = !{PY} -c "import importlib.metadata as m; print(m.version('qiskit-aer'))"
    ver = ver[-1].strip()
    !uv pip uninstall --quiet --python {PY} qiskit-aer
    !uv pip install --quiet --python {PY} "qiskit-aer-gpu-cu11=={ver}"
    !{PY} -c "from qiskit_aer import AerSimulator as S; print(S().available_devices())"

In [ ]:
# ---- benchmark (CPU variants always; GPU variants when Aer reports a GPU) -----------------
gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null
TAG = "colab-" + (gpu_name[0].split()[-1].lower() if HAS_GPU and gpu_name else "cpu")
!{PY} scripts/remote/bench_aer.py --tag {TAG} --rows {ROWS}

In [ ]:
# ---- package results (hashed manifest) and download ---------------------------------------
from google.colab import files

JOB_ID = f"bench-{TAG}"
ZIP = f"/content/{JOB_ID}.zip"
!{PY} scripts/remote/pack_results.py --job-dir results/remote/bench --job-id {JOB_ID} --zip {ZIP}
files.download(ZIP)
minutes = (time.time() - T0) / 60
print(f"elapsed {minutes:.1f} min; compute units = minutes / 60 x the rate in View resources")

## Back on the MacBook

```bash
uv run python scripts/remote/import_results.py ~/Downloads/bench-colab-<tag>.zip   # verified; never overwrites
uv run python scripts/remote/bench_aer.py --tag mac-m5pro                          # local reference (idle machine)
uv run python scripts/remote/compare_bench.py                                      # table + projections + agreement
```

Then commit the imported JSON (`results/remote/<job_id>/bench/*.json`) and the comparison table. The `.npz` reference arrays stay gitignored.